# Relación entre Mecánica de Fluidos y Sistemas Dinámicos
## Dinámica No Lineal, PINNs y SINDy

Este notebook explora la conexión entre la mecánica de fluidos y los sistemas dinámicos, con énfasis en:
- **Dinámica no lineal** en flujos de fluidos
- **PINNs (Physics-Informed Neural Networks)** para resolver ecuaciones de fluidos
- **SINDy (Sparse Identification of Nonlinear Dynamics)** para descubrir ecuaciones gobernantes

---

## 1. Introducción Teórica

### 1.1 Mecánica de Fluidos como Sistema Dinámico

Las ecuaciones de Navier-Stokes que gobiernan el flujo de fluidos pueden verse como un sistema dinámico infinito-dimensional:

$$\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u} \cdot \nabla)\mathbf{u} = -\frac{1}{\rho}\nabla p + \nu \nabla^2 \mathbf{u} + \mathbf{f}$$

$$\nabla \cdot \mathbf{u} = 0$$

donde:
- $\mathbf{u}$: campo de velocidades
- $p$: presión
- $\rho$: densidad
- $\nu$: viscosidad cinemática
- $\mathbf{f}$: fuerzas externas

### 1.2 Dinámica No Lineal en Fluidos

El término convectivo $(\mathbf{u} \cdot \nabla)\mathbf{u}$ introduce no linealidad, lo que lleva a fenómenos complejos como:
- Transición a turbulencia
- Caos determinista
- Bifurcaciones
- Estructuras coherentes

### 1.3 Reducción de Orden (ROM)

Podemos proyectar las ecuaciones de Navier-Stokes en un espacio de dimensión finita usando descomposición en modos propios (POD/DMD):

$$\mathbf{u}(\mathbf{x},t) \approx \sum_{i=1}^{r} a_i(t) \phi_i(\mathbf{x})$$

donde $a_i(t)$ son coeficientes temporales y $\phi_i(\mathbf{x})$ son modos espaciales.

Esto nos da un sistema dinámico finito-dimensional:

$$\frac{d\mathbf{a}}{dt} = \mathbf{f}(\mathbf{a})$$

### 1.4 SINDy: Identificación Dispersa de Dinámicas No Lineales

SINDy asume que las dinámicas pueden representarse como una combinación dispersa de funciones candidatas:

$$\frac{d\mathbf{a}}{dt} = \Theta(\mathbf{a})\Xi$$

donde $\Theta(\mathbf{a})$ es una matriz de funciones biblioteca y $\Xi$ es una matriz dispersa de coeficientes.

### 1.5 PINNs: Redes Neuronales Informadas por Física

Las PINNs incorporan las ecuaciones diferenciales directamente en la función de pérdida de la red neuronal:

$$L = L_{data} + \lambda L_{PDE}$$

donde $L_{PDE}$ mide cuánto viola la red las ecuaciones gobernantes.

## 2. Implementación Práctica

Importamos las librerías necesarias:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint, solve_ivp
from sklearn.linear_model import LinearRegression, Lasso
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("Librerías importadas exitosamente")

## 3. Ejemplo 1: Sistema de Lorenz como Modelo Reducido de Convección

El sistema de Lorenz es un modelo reducido de las ecuaciones de convección de Rayleigh-Bénard:

In [ ]:
def lorenz_system(state, t, sigma=10, rho=28, beta=8/3):
    """
    Sistema de Lorenz - modelo de convección en fluidos
    
    Parámetros:
    - sigma: número de Prandtl
    - rho: número de Rayleigh normalizado
    - beta: relación de aspecto
    """
    x, y, z = state
    
    dxdt = sigma * (y - x)
    dydt = x * (rho - z) - y
    dzdt = x * y - beta * z
    
    return [dxdt, dydt, dzdt]

# Condiciones iniciales
state0 = [1.0, 1.0, 1.0]
t = np.linspace(0, 50, 10000)

# Resolver el sistema
solution = odeint(lorenz_system, state0, t)
x, y, z = solution.T

# Visualizar el atractor de Lorenz
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(x, y, z, 'b-', lw=0.5, alpha=0.7)
ax.set_xlabel('X (Intensidad de convección)', fontsize=12)
ax.set_ylabel('Y (Diferencia de temperatura)', fontsize=12)
ax.set_zlabel('Z (Desviación del perfil lineal)', fontsize=12)
ax.set_title('Atractor de Lorenz - Modelo de Convección en Fluidos', fontsize=14)
plt.tight_layout()
plt.show()

# Series temporales
fig, axes = plt.subplots(3, 1, figsize=(12, 10))
axes[0].plot(t, x, 'r-', lw=1)
axes[0].set_ylabel('X')
axes[0].set_title('Series Temporales del Sistema de Lorenz')

axes[1].plot(t, y, 'g-', lw=1)
axes[1].set_ylabel('Y')

axes[2].plot(t, z, 'b-', lw=1)
axes[2].set_ylabel('Z')
axes[2].set_xlabel('Tiempo')

plt.tight_layout()
plt.show()

## 4. Ejemplo 2: Aplicación de SINDy al Sistema de Lorenz

Ahora aplicaremos SINDy para descubrir las ecuaciones del sistema de Lorenz a partir de los datos simulados.

In [ ]:
class SINDy:
    """
    Implementación simple de SINDy (Sparse Identification of Nonlinear Dynamics)
    """
    def __init__(self, threshold=0.1, max_iter=10):
        self.threshold = threshold
        self.max_iter = max_iter
        self.coefficients = None
        self.library_functions = None
    
    def create_library(self, X):
        """
        Crear biblioteca de funciones candidatas
        Incluye: términos constantes, lineales y cuadráticos
        """
        n_samples, n_features = X.shape
        
        # Biblioteca: [1, x, y, z, x², xy, xz, y², yz, z²]
        library = [np.ones(n_samples)]  # Término constante
        
        # Términos lineales
        for i in range(n_features):
            library.append(X[:, i])
        
        # Términos cuadráticos
        for i in range(n_features):
            for j in range(i, n_features):
                library.append(X[:, i] * X[:, j])
        
        return np.column_stack(library)
    
    def get_feature_names(self, n_features):
        """Obtener nombres de las características de la biblioteca"""
        names = ['1']
        
        features = ['x', 'y', 'z'][:n_features]
        
        # Lineales
        names.extend(features)
        
        # Cuadráticos
        for i in range(n_features):
            for j in range(i, n_features):
                names.append(f'{features[i]}{features[j]}')
        
        return names
    
    def fit(self, X, X_dot):
        """
        Ajustar el modelo SINDy usando regresión umbral iterativa
        """
        n_features = X.shape[1]
        Theta = self.create_library(X)
        self.library_functions = self.get_feature_names(n_features)
        
        n_library = Theta.shape[1]
        self.coefficients = np.zeros((n_library, n_features))
        
        # Para cada variable de estado
        for i in range(n_features):
            # Regresión lineal inicial
            coef = np.linalg.lstsq(Theta, X_dot[:, i], rcond=None)[0]
            
            # Umbral iterativo para dispersión
            for _ in range(self.max_iter):
                small_indices = np.abs(coef) < self.threshold
                coef[small_indices] = 0
                
                # Reajustar solo con términos no cero
                nonzero_indices = np.abs(coef) >= self.threshold
                if np.sum(nonzero_indices) > 0:
                    Theta_reduced = Theta[:, nonzero_indices]
                    coef_reduced = np.linalg.lstsq(Theta_reduced, X_dot[:, i], rcond=None)[0]
                    coef[nonzero_indices] = coef_reduced
            
            self.coefficients[:, i] = coef
        
        return self
    
    def print_equations(self):
        """Imprimir las ecuaciones descubiertas"""
        variables = ['x', 'y', 'z']
        
        for i, var in enumerate(variables):
            terms = []
            for j, coef in enumerate(self.coefficients[:, i]):
                if np.abs(coef) > 1e-6:
                    if j == 0:
                        terms.append(f"{coef:.3f}")
                    else:
                        sign = '+' if coef > 0 else '-'
                        terms.append(f"{sign} {np.abs(coef):.3f}{self.library_functions[j]}")
            
            equation = ' '.join(terms)
            print(f"d{var}/dt = {equation}")


# Calcular derivadas numéricas
dt = t[1] - t[0]
X_dot = np.gradient(solution, dt, axis=0)

# Aplicar SINDy
sindy = SINDy(threshold=0.1, max_iter=10)
sindy.fit(solution, X_dot)

print("Ecuaciones descubiertas por SINDy:")
print("="*50)
sindy.print_equations()

print("\nCoeficientes reales del sistema de Lorenz:")
print("dx/dt = 10.000(y - x) = -10.000x + 10.000y")
print("dy/dt = x(28 - z) - y = 28.000x - 1.000y - 1.000xz")
print("dz/dt = xy - (8/3)z = -2.667z + 1.000xy")

## 5. Ejemplo 3: Oscilador de Van der Pol como Modelo de Flujo Oscilatorio

El oscilador de Van der Pol modela fenómenos de flujo oscilatorio con amortiguamiento no lineal.

In [ ]:
def van_der_pol(state, t, mu=1.0):
    """
    Oscilador de Van der Pol
    Modela flujos con amortiguamiento no lineal
    """
    x, y = state
    
    dxdt = y
    dydt = mu * (1 - x**2) * y - x
    
    return [dxdt, dydt]

# Resolver para diferentes valores de mu
t_vdp = np.linspace(0, 30, 1000)
mu_values = [0.1, 1.0, 5.0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, mu in enumerate(mu_values):
    sol = odeint(van_der_pol, [2.0, 0.0], t_vdp, args=(mu,))
    x, y = sol.T
    
    axes[idx].plot(x, y, 'b-', lw=1.5, alpha=0.7)
    axes[idx].plot(x[0], y[0], 'ro', label='Inicio')
    axes[idx].set_xlabel('x (Desplazamiento)')
    axes[idx].set_ylabel('y (Velocidad)')
    axes[idx].set_title(f'Oscilador de Van der Pol (μ = {mu})')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Aplicar SINDy al oscilador de Van der Pol
sol_vdp = odeint(van_der_pol, [2.0, 0.0], t_vdp, args=(1.0,))
X_dot_vdp = np.gradient(sol_vdp, t_vdp[1] - t_vdp[0], axis=0)

sindy_vdp = SINDy(threshold=0.1, max_iter=10)
sindy_vdp.fit(sol_vdp, X_dot_vdp)

print("\nEcuaciones descubiertas para Van der Pol:")
print("="*50)
sindy_vdp.print_equations()

print("\nEcuaciones reales:")
print("dx/dt = y")
print("dy/dt = μ(1 - x²)y - x = μy - μx²y - x")

## 6. Ejemplo 4: Implementación Simple de PINN para la Ecuación de Burgers

La ecuación de Burgers es una simplificación de las ecuaciones de Navier-Stokes que captura la no linealidad y la difusión.

In [ ]:
# Intentar importar TensorFlow o PyTorch
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_AVAILABLE = True
    print("PyTorch disponible - ejecutando PINN completa")
except ImportError:
    try:
        import tensorflow as tf
        from tensorflow import keras
        TF_AVAILABLE = True
        print("TensorFlow disponible - ejecutando PINN completa")
    except ImportError:
        print("Ni PyTorch ni TensorFlow están disponibles.")
        print("Mostrando implementación conceptual de PINN.")
        TORCH_AVAILABLE = False
        TF_AVAILABLE = False

In [ ]:
if TORCH_AVAILABLE:
    # Definir la red neuronal
    class PINN(nn.Module):
        def __init__(self, layers):
            super(PINN, self).__init__()
            self.layers = nn.ModuleList()
            
            for i in range(len(layers) - 1):
                self.layers.append(nn.Linear(layers[i], layers[i+1]))
            
            self.activation = nn.Tanh()
        
        def forward(self, x):
            for i in range(len(self.layers) - 1):
                x = self.activation(self.layers[i](x))
            x = self.layers[-1](x)
            return x
    
    # Ecuación de Burgers: u_t + u*u_x = nu*u_xx
    def burgers_equation(u, x, t, nu=0.01):
        """Calcular el residuo de la ecuación de Burgers"""
        u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), 
                                   create_graph=True, retain_graph=True)[0]
        u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), 
                                   create_graph=True, retain_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), 
                                    create_graph=True, retain_graph=True)[0]
        
        return u_t + u * u_x - nu * u_xx
    
    # Generar datos de entrenamiento
    np.random.seed(42)
    n_points = 2000
    
    # Dominio espacial y temporal
    x_domain = np.linspace(-1, 1, 100)
    t_domain = np.linspace(0, 1, 50)
    X_grid, T_grid = np.meshgrid(x_domain, t_domain)
    
    # Solución exacta (para comparación)
    nu = 0.01
    U_exact = np.exp(-X_grid**2 / (4 * nu * (T_grid + 0.1))) / np.sqrt(T_grid + 0.1)
    
    # Puntos de collocation
    x_colloc = np.random.uniform(-1, 1, n_points)
    t_colloc = np.random.uniform(0, 1, n_points)
    
    # Convertir a tensores de PyTorch
    x_train = torch.FloatTensor(x_colloc).reshape(-1, 1)
    t_train = torch.FloatTensor(t_colloc).reshape(-1, 1)
    
    # Crear y entrenar la red
    layers = [2, 20, 20, 20, 1]  # Input: (x,t), Output: u
    model = PINN(layers)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Entrenamiento
    n_epochs = 5000
    loss_history = []
    
    print("Entrenando PINN para la ecuación de Burgers...")
    
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        
        # Requerir gradientes para las entradas
        x_train.requires_grad_(True)
        t_train.requires_grad_(True)
        
        # Predicción de la red
        inputs = torch.cat([x_train, t_train], dim=1)
        u_pred = model(inputs)
        
        # Calcular residuo de la PDE
        residual = burgers_equation(u_pred, x_train, t_train, nu)
        
        # Pérdida basada en la física
        loss_pde = torch.mean(residual**2)
        
        # Pérdida total
        loss = loss_pde
        
        loss.backward()
        optimizer.step()
        
        loss_history.append(loss.item())
        
        if epoch % 500 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.6f}")
    
    print("\nEntrenamiento completado!")
    
    # Visualizar resultados
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Historia de pérdida
    axes[0].plot(loss_history, 'b-', lw=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Historia de Pérdida durante el Entrenamiento')
    axes[0].grid(True, alpha=0.3)
    
    # Solución exacta
    im1 = axes[1].contourf(X_grid, T_grid, U_exact, levels=50, cmap='viridis')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('t')
    axes[1].set_title('Solución Exacta')
    plt.colorbar(im1, ax=axes[1])
    
    # Predicción de PINN (en puntos de la malla)
    with torch.no_grad():
        x_mesh = torch.FloatTensor(X_grid.flatten()).reshape(-1, 1)
        t_mesh = torch.FloatTensor(T_grid.flatten()).reshape(-1, 1)
        inputs_mesh = torch.cat([x_mesh, t_mesh], dim=1)
        u_pred_mesh = model(inputs_mesh).numpy().reshape(X_grid.shape)
    
    im2 = axes[2].contourf(X_grid, T_grid, u_pred_mesh, levels=50, cmap='viridis')
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('t')
    axes[2].set_title('Predicción PINN')
    plt.colorbar(im2, ax=axes[2])
    
    plt.tight_layout()
    plt.show()
    
    # Error relativo
    error = np.abs(U_exact - u_pred_mesh) / (np.abs(U_exact) + 1e-8)
    print(f"\nError relativo promedio: {np.mean(error):.6f}")
    print(f"Error relativo máximo: {np.max(error):.6f}")

else:
    # Implementación conceptual
    print("\n" + "="*70)
    print("IMPLEMENTACIÓN CONCEPTUAL DE PINN")
    print("="*70)
    
    print("""
Una Physics-Informed Neural Network (PINN) para la ecuación de Burgers:

Ecuación de Burgers:
    ∂u/∂t + u·∂u/∂x = ν·∂²u/∂x²

Arquitectura de la red:
    Input: (x, t) → [Red Neuronal] → Output: u(x,t)

Función de pérdida:
    L = L_data + λ·L_PDE
    
    donde:
    - L_data: Error en datos observados (si existen)
    - L_PDE: Residuo de la ecuación diferencial
    
    L_PDE = ||∂u/∂t + u·∂u/∂x - ν·∂²u/∂x²||²

Ventajas de PINNs en mecánica de fluidos:
    1. No requieren malla computacional
    2. Pueden incorporar datos experimentales
    3. Proporcionan soluciones continuas y diferenciables
    4. Útiles para problemas inversos (identificación de parámetros)
    5. Manejan naturalmente condiciones de frontera complejas

Aplicaciones típicas:
    - Simulación de flujos turbulentos
    - Reconstrucción de campos de flujo a partir de mediciones escasas
    - Identificación de parámetros de turbulencia
    - Optimización de diseño aerodinámico
    """)
    
    # Mostrar diagrama conceptual
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    
    # Diagrama de PINN
    text = """
    ╔══════════════════════════════════════════════════════════╗
    ║           ARQUITECTURA DE UNA PINN PARA FLUIDOS          ║
    ╠══════════════════════════════════════════════════════════╣
    ║                                                          ║
    ║  Entradas:      Red Neuronal:         Salida:           ║
    ║  ┌─────┐       ┌───────────┐        ┌──────┐            ║
    ║  │  x  │──────▶│           │───────▶│      │            ║
    ║  └─────┘       │           │        │      │            ║
    ║                │  Capas    │        │ u(x,t│  Campo de  ║
    ║  ┌─────┐       │  Ocultas  │        │      │  velocidad ║
    ║  │  t  │──────▶│           │───────▶│      │            ║
    ║  └─────┘       │           │        │      │            ║
    ║                │           │        └──────┘            ║
    ║                └───────────┘                             ║
    ║                     │                                    ║
    ║                     ▼                                    ║
    ║              ┌─────────────┐                             ║
    ║              │ Derivadas   │                             ║
    ║              │ Automáticas │                             ║
    ║              └─────────────┘                             ║
    ║                     │                                    ║
    ║                     ▼                                    ║
    ║              ┌─────────────┐                             ║
    ║              │  Residuo    │                             ║
    ║              │   PDE       │                             ║
    ║              │ ∂u/∂t +     │                             ║
    ║              │ u·∂u/∂x -   │                             ║
    ║              │ ν·∂²u/∂x²   │                             ║
    ║              └─────────────┘                             ║
    ║                                                          ║
    ╚══════════════════════════════════════════════════════════╝
    """
    ax.text(0.5, 0.5, text, fontsize=10, fontfamily='monospace', 
            verticalalignment='center', horizontalalignment='center')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

## 7. Comparación y Síntesis

### Tabla Comparativa de Métodos

In [ ]:
import pandas as pd

# Crear tabla comparativa
data = {
    'Método': ['Sistemas Dinámicos Clásicos', 'SINDy', 'PINNs'],
    'Enfoque': ['Analítico/Numerico', 'Descubrimiento de ecuaciones', 'Aprendizaje automático'],
    'Requiere Ecuaciones': ['Sí', 'No', 'Sí (como constraint)'],
    'Requiere Datos': ['No', 'Sí', 'Opcional'],
    'Interpretabilidad': ['Alta', 'Alta', 'Media-Baja'],
    'Escalabilidad': ['Limitada', 'Buena', 'Excelente'],
    'Aplicación Principal': ['Análisis cualitativo', 'Modelado reducido', 'Simulación directa']
}

df_comparison = pd.DataFrame(data)
print(df_comparison.to_string(index=False))

# Visualizar comparación
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

table = ax.table(cellText=df_comparison.values,
                 colLabels=df_comparison.columns,
                 cellLoc='center',
                 loc='center',
                 colColours=['#3498db'] * len(df_comparison.columns))

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

ax.set_title('Comparación de Métodos para Mecánica de Fluidos', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

## 8. Conclusiones

### Puntos Clave:

1. **Conexión Fundamental**: La mecánica de fluidos puede entenderse como un sistema dinámico infinito-dimensional que puede reducirse a sistemas de dimensión finita.

2. **Dinámica No Lineal**: El término convectivo en las ecuaciones de Navier-Stokes introduce no linealidad, llevando a comportamientos complejos como turbulencia y caos.

3. **SINDy**: Permite descubrir ecuaciones gobernantes a partir de datos, proporcionando modelos interpretables y parsimoniosos.

4. **PINNs**: Combina el poder del aprendizaje profundo con conocimiento físico, permitiendo resolver PDEs sin malla y manejar problemas inversos.

5. **Sinergia**: Estos métodos se complementan:
   - SINDy puede identificar modelos reducidos
   - PINNs puede resolver las ecuaciones identificadas
   - El análisis de sistemas dinámicos proporciona comprensión teórica

### Aplicaciones Futuras:

- Control de flujos turbulentos
- Diseño optimizado de aeronaves
- Predicción meteorológica mejorada
- Modelado cardiovascular
- Energías renovables (eólica, hidroeléctrica)

## 9. Referencias

1. Brunton, S. L., Proctor, J. L., & Kutz, J. N. (2016). Discovering governing equations from data by sparse identification of nonlinear dynamical systems. PNAS.

2. Raissi, M., Perdikaris, P., & Karniadakis, G. E. (2019). Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. JCP.

3. Strogatz, S. H. (2018). Nonlinear dynamics and chaos with student solutions manual. CRC Press.

4. Holmes, P., Lumley, J. L., Berkooz, G., & Rowley, C. W. (2012). Turbulence, coherent structures, dynamical systems and symmetry. Cambridge University Press.

5. Kutz, J. N., Brunton, S. L., Brunton, B. W., & Proctor, J. L. (2016). Dynamic mode decomposition: data-driven modeling of complex systems. SIAM.

---
**Nota**: Este notebook es educativo y demostrativo. Para aplicaciones reales, se recomienda usar implementaciones más robustas y validadas de SINDy (como la librería `pysindy`) y PINNs (como `DeepXDE` o `NeuroDiffEq`).